## 历史构造结果回放

本 Notebook 展示保存的历史摘要和日期审计，不训练模型、不重跑账本，也不重建第39章逐行预测。未随包提供真实市场数据。

[本语言逐本运行模式与准备条件](../../../NOTEBOOK_GUIDE.md)。运行步骤、数据要求和预期输出见本指南。

# 项目6基础管线：历史ML结果读者

历史设计为标签→日期前推→训练内调参→预测→美元中性权重→29章账本及费用。当前只回放base/regime结果，不是从配置冷启动运行该流程。

信号 $D_k$ 收盘，成交/标签起点 $D_{k+1}$，标签终点/可得 $D_{k+6}$（本构造设计）。每5信号日调仓、NAV逐日记账，不把5日标签当日收益。

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("p06_baseline_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import json, numpy as np, pandas as pd
from IPython.display import display, Image
FIX = ROOT / "fixtures/projects/project06"
FROZEN = json.loads((FIX / "ch39_results.json").read_text())
AUDIT = json.loads((FIX / "display_recompute.json").read_text())
P06_BASE = json.loads((FIX / "p06_base_summary.json").read_text())
P06_REGIME = json.loads((FIX / "p06_regime_summary.json").read_text())
print("Historical accepted CONSTRUCTED display replay; no training, no Ch39 row reconstruction; real route blocked")


## 1. 冻结设计与日期契约（不调用pp.run）

In [ ]:
print('Historical frozen panel, signal closeDk, trade/label startDk+1, label endDk+6; real data blocked.')

## 2. 自身有效日期、共同配对日期与IC

每情景1002候选标签日。t停止推断展示，旧dropna压缩NW(4)不等于四交易日HAC。基准常数IC未定义；P06不能替代39缺失的历史行。

In [ ]:
for scenario, metrics in AUDIT['p06_v1'].items():
 print(scenario)
 display(pd.DataFrame(metrics).T[['own_valid_dates','eligible_label_dates','constant_prediction_dates','own_mean','paired_ridge_dates','paired_model_mean','ridge_on_paired_mean','paired_difference_mean','t_display']])

## 3. 组合层：成本前、单边 10 与 20 个基点

零预测与训练均值空仓（NAV 恒为初始资金）。负结果保留。

In [ ]:
for scenario,s in [('base',P06_BASE),('regime',P06_REGIME)]:
 print(scenario, 'Historical frozen portfolio results; negative outcomes retained.')
 display(pd.DataFrame({m:{fee:values for fee,values in by_fee.items()} for m,by_fee in s['portfolio_metrics'].items()}).T)

## 4. 每折选出的超参数（训练内选择，测试只评价一次）

In [ ]:
print("Chosen parameters remain in historical fold_records; not bundled or retuned. This reader replay does not reconstruct the missing Ch39 historical prediction rows.")


## 5. 成本后 NAV

In [ ]:
for scenario in ['base','regime']:
 display(Image(filename=str(FIX/f'p06_nav_{scenario}.png')))
print('Frozen NAV plots; no new ledger/backtest.')

完整问答：基础Lasso自身630日不能直接减Ridge1002日；共同630日上的描述差为+0.00172429，不附新推断t。39无冻结行不可替代；不重训补造。